# Traffic Cone YOLOv8 Training
This notebook is pre-configured to train a YOLOv8 model using the Traffic Cone dataset split.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

### 1. Set the Dataset Path
Update the `ROOT_DIR` below to point to your `traffic_cone_split` folder on Google Drive.
**Tip:** Use the folder icon on the left to find your folder, right-click `data.yaml`, and select 'Copy path'.

In [ ]:
# Example: '/content/gdrive/MyDrive/traffic_cone_split'
ROOT_DIR = '/content/gdrive/MyDrive/traffic_cone_split'

import os
if not os.path.exists(ROOT_DIR):
    print(f"❌ ERROR: Path not found: {ROOT_DIR}")
    print("Please check your Google Drive folder name and update ROOT_DIR.")
else:
    print(f"✅ Found directory: {ROOT_DIR}")
    !ls {ROOT_DIR}

### 2. Auto-Fix data.yaml Paths
This cell ensures the paths inside `data.yaml` are correctly formatted for YOLO.

In [ ]:
import yaml
yaml_path = os.path.join(ROOT_DIR, 'data.yaml')

if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Fix relative paths
    config['train'] = 'train/images'
    config['val'] = 'valid/images'
    config['test'] = 'test/images'
    
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f)
    
    print("✅ data.yaml has been verified and updated!")
    print(config)
else:
    print(f"❌ ERROR: {yaml_path} not found!")

In [ ]:
!pip install ultralytics
!pip install onnx
!pip install onnx_tf
!pip install tensorflow

In [ ]:
from ultralytics import YOLO
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load a pretrained YOLOv8n model
model = YOLO("yolov8n.yaml")

# Train the model
results = model.train(data=os.path.join(ROOT_DIR, "data.yaml"), epochs=80, device=device)

### Validation
Evaluate the model's performance on the validation set.

In [ ]:
best_model_path = os.path.join(results.save_dir, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

val_metrics = best_model.val(data=os.path.join(ROOT_DIR, 'data.yaml'), split='val', name='traffic_cone_val')

print('Validation metrics:')
print(val_metrics)

### Auto-Save to Google Drive
Run this to save all results (weights, plots, etc.) permanently.

In [ ]:
import shutil
drive_results_path = os.path.join(ROOT_DIR, 'results')

shutil.copytree(
    '/content/runs/detect', 
    drive_results_path, 
    dirs_exist_ok=True
)

print(f"Success! Everything has been synced to {drive_results_path}")